# Generasi jawaban Qwen3.8-27B (Subbab 4.10)
Prompt sudah disusun lokal (instruksi, konteks [1]–[5], pertanyaan). Notebook ini hanya membangkitkan jawaban:
*greedy decoding*, maksimum 1.024 token, mode berpikir dinonaktifkan. Prompt identik antarmetode dibangkitkan sekali.

In [ ]:
MODE = "full"            # "smoke" = 5 unit x 3 metode, "full" = seluruh prompt unik
MODEL_ID = "RedHatAI/Qwen3.8-27B-INT4"
MAX_NEW_TOKENS = 1024
BATCH = 512
TIME_BUDGET_S = 11 * 3600    # berhenti memulai batch baru setelah 11 jam (batas sesi 12 jam)

In [ ]:
!pip install -q -U vllm 2>&1 | tail -2

In [ ]:
import glob, json, os, platform, subprocess, time
import torch, vllm
from huggingface_hub import model_info
t_start = time.time()
env = {"vllm": vllm.__version__, "torch": torch.__version__, "python": platform.python_version(),
       "gpus": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
       "model_id": MODEL_ID, "model_revision": model_info(MODEL_ID).sha, "mode": MODE}
print(json.dumps(env, indent=2))
json.dump(env, open("/kaggle/working/environment.json", "w"), indent=2)

In [ ]:
def find(name):
    hits = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True))
    return hits
prompts = [json.loads(l) for l in open(find("prompts.jsonl")[0], encoding="utf-8")]
if MODE == "smoke":
    keep = set(json.load(open(find("smoke_pids.json")[0])))
    prompts = [p for p in prompts if p["pid"] in keep]
done = set()
for path in find("answers.jsonl"):          # keluaran sesi sebelumnya (bila dilampirkan) untuk melanjutkan
    done |= {json.loads(l)["pid"] for l in open(path, encoding="utf-8")}
out_path = "/kaggle/working/answers.jsonl"
if os.path.exists(out_path):
    done |= {json.loads(l)["pid"] for l in open(out_path, encoding="utf-8")}
todo = sorted((p for p in prompts if p["pid"] not in done), key=lambda p: p["prompt"])
print(f"prompt: {len(prompts)}, sudah selesai: {len(done & {p['pid'] for p in prompts})}, sisa: {len(todo)}")

In [ ]:
from vllm import LLM, SamplingParams
llm = LLM(model=MODEL_ID, tensor_parallel_size=torch.cuda.device_count(), dtype="float16",
          max_model_len=4096, gpu_memory_utilization=0.92, seed=0, enable_prefix_caching=True,
          kv_cache_dtype="float16",            # T4 (SM75) tidak mendukung KV cache FP8 bawaan model
          max_num_batched_tokens=2048, max_num_seqs=32, enforce_eager=True,   # muat di 2x15 GB
          disable_custom_all_reduce=True, limit_mm_per_prompt={"image": 0, "video": 0})
sp = SamplingParams(temperature=0.0, max_tokens=MAX_NEW_TOKENS, seed=0)   # greedy
CHAT_KW = {"enable_thinking": False}                                       # mode berpikir dimatikan

In [ ]:
with open(out_path, "a", encoding="utf-8") as f:
    for i in range(0, len(todo), BATCH):
        if time.time() - t_start > TIME_BUDGET_S:
            print("batas waktu sesi tercapai; lanjutkan di sesi berikutnya"); break
        batch = todo[i:i + BATCH]
        t0 = time.time()
        outs = llm.chat([[{"role": "user", "content": p["prompt"]}] for p in batch], sp,
                        chat_template_kwargs=CHAT_KW, use_tqdm=False)
        dt = time.time() - t0
        for p, o in zip(batch, outs):
            c = o.outputs[0]
            f.write(json.dumps({"pid": p["pid"], "answer": c.text, "n_output_tokens": len(c.token_ids),
                                "n_prompt_tokens": len(o.prompt_token_ids), "finish_reason": c.finish_reason,
                                "batch_seconds": round(dt, 3), "batch_size": len(batch),
                                "seconds_per_prompt": round(dt / len(batch), 3)}, ensure_ascii=False) + "\n")
        f.flush()
        print(f"{min(i + BATCH, len(todo))}/{len(todo)}  {dt:.0f}s  total {(time.time() - t_start) / 60:.0f} menit", flush=True)

In [ ]:
rows = [json.loads(l) for l in open(out_path, encoding="utf-8")]
think = sum("<think>" in r["answer"] for r in rows)
trunc = sum(r["finish_reason"] == "length" for r in rows)
print(f"jawaban: {len(rows)}, memuat <think>: {think}, terpotong 1.024 token: {trunc}")
for r in rows[:5]:
    print("-" * 80); print(r["pid"], r["n_output_tokens"], "token"); print(r["answer"][:800])